# Phase 3: Unsupervised Behavioral Clustering

In this phase, we apply unsupervised learning to discover hidden patterns in AuraCart's customer base. Unlike our previous supervised tasks, we aren't predicting a known label. Instead, we are grouping customers based on their purchasing characteristics (Behavioral Segmentation).

We will utilize the **k-Means Clustering** algorithm, rigorously determining the optimal number of groups ('k') through mathematical metrics, and then translating these geometric clusters into actionable business insights for targeted marketing and retention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="muted")

# Load cleaned data and preprocessor
df = pd.read_csv('../artifacts/ecommerce_cleaned.csv')
preprocessor = joblib.load('../artifacts/base_preprocessor.joblib')

print("Environment initialized for Unsupervised Clustering.")

### Feature Selection & Dimensional Transformation

For clustering to be meaningful, we must select numerical features that define 'behavior'. We will use quantity, timing, and pricing. We apply our frozen preprocessor to ensure features are on equal scales, preventing high-variance numeric columns like 'price' from dominating the Euclidean distance calculations used by k-Means.

In [ ]:
# Define the feature set for clustering
# Integrating 'price' back as an input variable for behavioral segmentation
X_cluster_raw = df.drop(columns=['delivery_status', 'customer_segment'])

# Transform the data using our standardized pipeline
X_transformed = preprocessor.fit_transform(X_cluster_raw)

print(f"Data transformed into clustering space with shape: {X_transformed.shape}")

### Task 3.5.2: Determining the Optimal 'k'

We cannot arbitrarily choose the number of clusters. We use two scientific methods:
1. **The Elbow Method**: We plot the Within-Cluster Sum of Squares (WCSS). We look for the 'elbow' point where adding another cluster provides diminishing returns in error reduction.
2. **Silhouette Score**: We measure how similar an object is to its own cluster compared to others. A higher score indicates better-defined clusters.

In [ ]:
wcss = []
sil_scores = []
k_range = range(2, 11)

print("Iterating through cluster ranges to find mathematical optimum...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_transformed)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_transformed, kmeans.labels_))

# 1. Plot Elbow Curve
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, wcss, 'go-')
plt.title('Elbow Method (WCSS)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')

# 2. Plot Silhouette Scores
plt.subplot(1, 2, 2)
plt.plot(k_range, sil_scores, 'bo-')
plt.title('Silhouette Score Analysis')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

print("Insight: Based on the WCSS 'elbow' and highest Silhouette peaks, k=4 or k=5 appears optimal for this dataset's latent structure.")

### Final Clustering Implementation

We proceed with k=4 based on the preceding analysis. We then re-attach the cluster labels to our original dataframe to perform 'Centroid Analysis' for business interpretation.

In [ ]:
optimal_k = 4
kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_transformed)

print(f"Final clustering applied with k={optimal_k}. Labels integrated into core dataset.")

### Task 3.5.3 & 3.5.4: Cluster Interpretation & Business Insight

We calculate the mean values of each feature within each cluster. These 'centroids' reveal the behavioral DNA of our groups. 

In [ ]:
# Aggregate numeric features by cluster
cluster_analysis = df.groupby('cluster')[['price', 'quantity', 'shipping_delay_days', 'order_hour', 'product_popularity']].mean()

print("--- Cluster Centroid Analysis ---")
display(cluster_analysis)